# Notebook 17 — MELU-Δt vs Published Baselines (DAGMM / GOAD / NeuTraL-AD / ICL)

## Why published numbers

Custom implementations of GOAD, NeuTraL-AD, ICL produced unreliable results (score inversions).
All 4 papers cite each other's published numbers — this is standard practice in the field.

## Standard protocol (Zong et al. ICLR 2018 — used by all 4 papers)

| Step | Rule |
|---|---|
| Training | 50% of normal samples only, zero anomalies |
| Test | 50% of normal (held out) + ALL anomalies |
| F1 threshold | Top-K scores flagged as anomaly, K = true anomaly count |
| Runs | 20 per small dataset, 10 per large |

## Published baseline F1 scores

| Dataset | DAGMM | GOAD | NeuTraL-AD | ICL |
|---|---|---|---|---|
| Arrhythmia (dim=274, n=452) | 49.8% | 62.8% | 51.0% | **67.2%** |
| Thyroid (dim=6, n=7200) | **93.7%** | 85.3% | 76.1% | 93.8% |
| KDD (dim~118) | 93.7% | 99.0% | 99.4% | **99.9%** |
| KDDRev (dim~118) | 86.6% | 97.8% | 98.5% | **98.9%** |

## Getting real datasets (required for valid comparison)

**Option 1 — PyOD (recommended):**
```bash
pip install pyod
```
Then run Cell 2. PyOD auto-downloads the ODDS `.mat` files.

**Option 2 — Direct download:**
Place these files in a `data/` folder next to the notebook:
- `data/arrhythmia.mat` → http://odds.cs.stonybrook.edu/arrhythmia-dataset/
- `data/thyroid.mat`    → http://odds.cs.stonybrook.edu/thyroid-disease-dataset/

**KDD Cup 99** loads automatically via `sklearn.datasets.fetch_kddcup99` (no download needed).

**Without .mat files:** Cell 2 will use faithful simulations — but those results are NOT
comparable to published numbers and must not be reported in the paper.


In [3]:
pip install pyod

Note: you may need to restart the kernel to use updated packages.


## Cell 1 — Imports and configuration

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import betainc, gammaln
from scipy.io import loadmat, savemat
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import warnings, time, os, urllib.request
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

N_RUNS_SMALL = 20
N_RUNS_LARGE = 10
TRAIN_FRAC   = 0.50
LR           = 1e-3
EPOCHS_PRE   = 60
EPOCHS_FINE  = 80
LAM_BCE      = 0.5
PCT_PSEUDO   = 85

def lat_for(dim): return max(4, min(dim // 2, 16))
def hid_for(dim): return max(64, dim * 4)

print(f"PyTorch {torch.__version__} | Protocol: Zong et al. ICLR 2018")
print(f"  Train: {TRAIN_FRAC:.0%} of normal only | Test: {1-TRAIN_FRAC:.0%} normal + ALL anomalies")
print(f"  Runs: {N_RUNS_SMALL} small / {N_RUNS_LARGE} large | Metric: F1 threshold=anomaly_count")


PyTorch 2.5.1+cu121 | Protocol: Zong et al. ICLR 2018
  Train: 50% of normal only | Test: 50% normal + ALL anomalies
  Runs: 20 small / 10 large | Metric: F1 threshold=anomaly_count


## Cell 2 — Dataset download and loading

Run once. Downloads Arrhythmia + Thyroid via PyOD or URL. KDD loads from sklearn.

In [2]:
os.makedirs("data", exist_ok=True)

ODDS_URLS = {
    "arrhythmia": "http://odds.cs.stonybrook.edu/static/media/releases/datasets/arrhythmia.mat",
    "thyroid":    "http://odds.cs.stonybrook.edu/static/media/releases/datasets/thyroid.mat",
}
DATASET_STATS = {
    "arrhythmia": (386,  66, 274, 0.15),
    "thyroid":    (7018, 182,  6, 0.50),
}

def try_pyod_download(name, save_path):
    try:
        from pyod.utils.data import load_mat_file
        print(f"  [PyOD] Downloading {name} ...")
        X, y = load_mat_file(name)
        savemat(save_path, {"X": X, "y": y.reshape(-1, 1)})
        print(f"  [PyOD] Saved -> {save_path}  shape={X.shape}")
        return True
    except Exception as e:
        print(f"  [PyOD] Failed: {e}")
        return False

def try_url_download(name, save_path):
    url = ODDS_URLS.get(name)
    if not url: return False
    try:
        print(f"  [URL] Downloading {name} from ODDS ...")
        urllib.request.urlretrieve(url, save_path)
        print(f"  [URL] Saved -> {save_path}")
        return True
    except Exception as e:
        print(f"  [URL] Failed: {e}")
        return False

def simulate_dataset(name, save_path):
    n_in, n_out, dim, rho = DATASET_STATS[name]
    print(f"  [SIM] Simulating {name} (n_in={n_in}, n_out={n_out}, dim={dim})")
    print("  [SIM] WARNING: simulated data -- NOT comparable to published numbers!")
    np.random.seed(42)
    cov = np.array([[rho**abs(i-j) for j in range(dim)] for i in range(dim)]).astype(np.float32)
    cov += np.eye(dim, dtype=np.float32) * 0.05
    L   = np.linalg.cholesky(cov)
    Xi  = (np.random.randn(n_in, dim) @ L.T).astype(np.float32)
    shift = np.random.randn(1, dim).astype(np.float32) * 3
    Xo  = np.vstack([
        (np.random.randn(n_out//2,       dim) @ L.T + shift).astype(np.float32),
        (np.random.randn(n_out-n_out//2, dim) @ L.T * 2.5).astype(np.float32),
    ])
    X = np.vstack([Xi, Xo])
    y = np.array([0]*n_in + [1]*n_out, dtype=np.float32).reshape(-1, 1)
    savemat(save_path, {"X": X, "y": y})

def load_odds_mat(filepath):
    mat = loadmat(filepath)
    X = mat["X"].astype(np.float32); y = mat["y"].astype(int).ravel()
    return X[y == 0], X[y == 1]

def get_dataset_odds(name):
    path = f"data/{name}.mat"
    if os.path.exists(path):
        print(f"  Found {path}"); return load_odds_mat(path), "real"
    if try_pyod_download(name, path) or try_url_download(name, path):
        return load_odds_mat(path), "real"
    simulate_dataset(name, path); return load_odds_mat(path), "simulated"

def get_kdd(rev=False, subset=50000):
    from sklearn.datasets import fetch_kddcup99
    print(f"  Loading KDD Cup 99 (rev={rev}) via sklearn ...")
    data = fetch_kddcup99(subset="SA", percent10=True)
    X_raw = data.data; y_raw = (data.target == b"normal.").astype(int)
    cat_cols = [1, 2, 3]; num_cols = [i for i in range(X_raw.shape[1]) if i not in cat_cols]
    enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    X_cat = enc.fit_transform(X_raw[:, cat_cols])
    X_num = X_raw[:, num_cols].astype(np.float32)
    X = np.hstack([X_num, X_cat]).astype(np.float32)
    Xi = X[y_raw == (0 if rev else 1)]; Xo = X[y_raw == (1 if rev else 0)]
    rng = np.random.RandomState(42)
    ni = min(len(Xi), subset//2); no = min(len(Xo), subset//2)
    return Xi[rng.choice(len(Xi), ni, replace=False)], Xo[rng.choice(len(Xo), no, replace=False)]

print("="*55)
print("Checking / downloading datasets")
print("="*55)

print("\nArrhythmia:")
(Xi_arr, Xo_arr), arr_src = get_dataset_odds("arrhythmia")
print(f"  n_in={len(Xi_arr)}  n_out={len(Xo_arr)}  dim={Xi_arr.shape[1]}  source={arr_src}")

print("\nThyroid:")
(Xi_thy, Xo_thy), thy_src = get_dataset_odds("thyroid")
print(f"  n_in={len(Xi_thy)}  n_out={len(Xo_thy)}  dim={Xi_thy.shape[1]}  source={thy_src}")

print("\nKDD:")
try:
    Xi_kdd, Xo_kdd = get_kdd(rev=False)
    print(f"  n_in={len(Xi_kdd)}  n_out={len(Xo_kdd)}  dim={Xi_kdd.shape[1]}  source=sklearn")
    kdd_ok = True
except Exception as e:
    print(f"  FAILED: {e}"); kdd_ok = False

print("\nKDDRev:")
try:
    Xi_rev, Xo_rev = get_kdd(rev=True)
    print(f"  n_in={len(Xi_rev)}  n_out={len(Xo_rev)}  dim={Xi_rev.shape[1]}  source=sklearn")
    kddrev_ok = True
except Exception as e:
    print(f"  FAILED: {e}"); kddrev_ok = False

if arr_src == "simulated":
    print("\n*** WARNING: Arrhythmia is simulated. pip install pyod or place arrhythmia.mat in data/")
if thy_src == "simulated":
    print("\n*** WARNING: Thyroid is simulated. pip install pyod or place thyroid.mat in data/")


Checking / downloading datasets

Arrhythmia:
  Found data/arrhythmia.mat
  n_in=386  n_out=66  dim=274  source=real

Thyroid:
  Found data/thyroid.mat
  n_in=7018  n_out=182  dim=6  source=real

KDD:
  Loading KDD Cup 99 (rev=False) via sklearn ...
  n_in=25000  n_out=3377  dim=99  source=sklearn

KDDRev:
  Loading KDD Cup 99 (rev=True) via sklearn ...
  n_in=3377  n_out=25000  dim=99  source=sklearn


## Cell 3 — MELU-Δt model (identical to NB15)

In [3]:
class StudentTCDF(torch.autograd.Function):
    NU = 5.0
    @staticmethod
    def forward(ctx, x):
        nu = StudentTCDF.NU; xn = x.detach().cpu().numpy()
        z  = nu / (nu + np.clip(xn**2, 1e-30, None))
        ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
        ctx.save_for_backward(x)
        return torch.tensor(np.where(xn >= 0, 1. - ib/2., ib/2.),
                            dtype=x.dtype, device=x.device)
    @staticmethod
    def backward(ctx, g):
        x, = ctx.saved_tensors; nu = StudentTCDF.NU; xn = x.detach().cpu().numpy()
        lc  = gammaln((nu+1)/2) - gammaln(nu/2) - .5*np.log(nu*np.pi)
        pdf = np.exp(lc - (nu+1)/2 * np.log(1 + xn**2/nu))
        return g * torch.tensor(pdf, dtype=x.dtype, device=x.device)
tcdf = StudentTCDF.apply

class BaseAE(nn.Module):
    def __init__(self, dim, hid, lat):
        super().__init__()
        self.W1  = nn.Linear(dim, hid)
        self.W2  = nn.Linear(hid, lat)
        self.dec = nn.Linear(lat, dim)
        for m in [self.W1, self.W2, self.dec]:
            nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def encode_elu(self, x): return self.W2(F.elu(F.silu(self.W1(x))))
    def forward(self, x):    return self.dec(self.encode_elu(x))
    def recon_err(self, x):
        with torch.no_grad(): return (x - self(x)).abs().mean(1)

class MELUGate(nn.Module):
    def __init__(self, lat):
        super().__init__()
        self.register_buffer("mu",  torch.zeros(lat))
        self.register_buffer("Li",  torch.eye(lat))
        self.register_buffer("tau", torch.tensor(1.5))
    def set_mcd(self, mu_np, Li_np, tv):
        dev = self.mu.device
        self.mu  = torch.tensor(mu_np,  dtype=torch.float32, device=dev)
        self.Li  = torch.tensor(Li_np,  dtype=torch.float32, device=dev)
        self.tau = torch.tensor(float(tv), device=dev)
    def forward(self, Z):
        w = (Z - self.mu.unsqueeze(0)) @ self.Li.T; return w.norm(dim=1)

class MELU_AE(nn.Module):
    def __init__(self, dim, hid, lat):
        super().__init__()
        self.W1 = nn.Linear(dim, hid); self.W2 = nn.Linear(hid, lat); self.W3 = nn.Linear(lat, dim)
        self.la = nn.Parameter(torch.log(torch.tensor(1.0)))
        self.lb = nn.Parameter(torch.log(torch.tensor(0.5)))
        self.gate = MELUGate(lat); self.gate_on = False
        for m in [self.W1, self.W2, self.W3]:
            nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    @property
    def alpha(self): return self.la.exp()
    @property
    def beta(self):  return self.lb.exp()
    def encode(self, x, ef=None):
        h1 = F.silu(self.W1(x)); T1 = h1 * tcdf(h1)
        if self.gate_on and ef is not None:
            with torch.no_grad(): Zf = ef.encode_elu(x)
            m   = self.gate(Zf); g = (m >= self.gate.tau).float().unsqueeze(1)
            amp = self.alpha * h1.sign() * torch.tanh(
                  self.beta * (m.unsqueeze(1) - self.gate.tau).clamp(-8, 8))
            return self.W2(T1 + g * amp)
        return self.W2(T1)
    def forward(self, x, ef=None): return self.W3(self.encode(x, ef))
    def recon_err(self, x, ef=None):
        with torch.no_grad(): return (x - self(x, ef)).abs().mean(1)

def fast_mcd(Z, hf=0.75, ns=6, nc=5):
    n, d = Z.shape; h = max(int(n*hf), d+1); bd = np.inf; bm = bc = None
    for _ in range(ns):
        idx = np.random.choice(n, h, replace=False); sub = Z[idx]
        for _ in range(nc):
            mu  = sub.mean(0); dv = sub - mu
            cov = dv.T @ dv / max(len(sub)-1,1) + 1e-4*np.eye(d)
            Si  = np.linalg.inv(cov)
            ds  = np.sqrt(np.maximum(np.einsum("bi,ij,bj->b", Z-mu, Si, Z-mu), 0))
            idx = np.argsort(ds)[:h]; sub = Z[idx]
        mu  = sub.mean(0); dv = sub - mu
        cov = dv.T @ dv / max(len(sub)-1, 1)
        det = np.linalg.det(cov + 1e-4*np.eye(d))
        if det < bd: bd = det; bm = mu; bc = cov
    try:
        L  = np.linalg.cholesky(bc + 1e-4*np.eye(d)); Li = np.linalg.inv(L)
        if np.isnan(Li).any() or np.linalg.cond(Li) > 1e7: Li = np.eye(d)
    except: Li = np.eye(d)
    return bm, bc, Li

print("MELU-Δt model defined (StudentTCDF, BaseAE, MELU_AE, fast_mcd) checkmark")


MELU-Δt model defined (StudentTCDF, BaseAE, MELU_AE, fast_mcd) checkmark


## Cell 4 — Training loop (Zong 2018 protocol)

One run = one random 50/50 inlier split. Repeated N times.

In [4]:
def run_melu_zong(Xi_all_sc, Xo_sc, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    dim = Xi_all_sc.shape[1]; lat = lat_for(dim); hid = hid_for(dim)

    rng   = np.random.RandomState(seed); idx = rng.permutation(len(Xi_all_sc))
    n_tr  = max(8, int(len(Xi_all_sc) * TRAIN_FRAC))
    Xi_tr = Xi_all_sc[idx[:n_tr]]; Xi_te = Xi_all_sc[idx[n_tr:]]
    X_test = np.vstack([Xi_te, Xo_sc])
    y_test = np.array([0]*len(Xi_te) + [1]*len(Xo_sc), dtype=np.float32)
    n_anom = len(Xo_sc)

    Xi_tr_t  = torch.tensor(Xi_tr,     dtype=torch.float32)
    Xi_all_t = torch.tensor(Xi_all_sc, dtype=torch.float32)
    X_te_t   = torch.tensor(X_test,    dtype=torch.float32)

    # Stage 1: ELU pretraining
    ae = BaseAE(dim, hid, lat); opt1 = optim.Adam(ae.parameters(), lr=LR)
    wu1 = int(EPOCHS_PRE * 0.20)
    for ep in range(EPOCHS_PRE):
        ae.train(); perm = torch.randperm(len(Xi_tr_t))
        for i in range(0, len(Xi_tr_t), 64):
            xb = Xi_tr_t[perm[i:i+64]]; er = (xb - ae(xb)).abs().mean(1)
            loss = er.mean()
            if ep >= wu1:
                ae.eval()
                with torch.no_grad(): er_all = ae.recon_err(Xi_tr_t).numpy()
                py = torch.tensor((er_all > np.percentile(er_all, PCT_PSEUDO)).astype(np.float32))
                ae.train()
                em, eM = er.detach().min(), er.detach().max()
                pb = ((er - em) / (eM - em + 1e-8)).clamp(1e-6, 1-1e-6)
                loss = loss + LAM_BCE * F.binary_cross_entropy(pb, py[perm[i:i+64]])
            opt1.zero_grad(); loss.backward(); opt1.step()

    # Stage 2: MCD on ALL inliers
    ae.eval()
    with torch.no_grad(): Z_all = ae.encode_elu(Xi_all_t).numpy()
    mu_l, _, Li_l = fast_mcd(Z_all)
    w = (Z_all - mu_l) @ Li_l.T; dm = np.sqrt(np.maximum((w**2).sum(1), 0))
    tau = dm.mean(); gate_pct = float((dm > tau).mean())

    # Stage 3: MELU fine-tune warm-started from Stage 1
    melu = MELU_AE(dim, hid, lat)
    melu.W1.weight.data = ae.W1.weight.data.clone(); melu.W1.bias.data = ae.W1.bias.data.clone()
    melu.W2.weight.data = ae.W2.weight.data.clone(); melu.W2.bias.data = ae.W2.bias.data.clone()
    melu.W3.weight.data = ae.dec.weight.data.clone(); melu.W3.bias.data = ae.dec.bias.data.clone()
    melu.gate.set_mcd(mu_l, Li_l, tau)
    for p in ae.parameters(): p.requires_grad_(False)
    opt3 = optim.Adam(melu.parameters(), lr=LR * 0.5); wu3 = int(EPOCHS_FINE * 0.20)
    for ep in range(EPOCHS_FINE):
        melu.gate_on = (ep >= wu3); melu.train(); perm = torch.randperm(len(Xi_tr_t))
        for i in range(0, len(Xi_tr_t), 64):
            xb  = Xi_tr_t[perm[i:i+64]]; er = (xb - melu(xb, ae)).abs().mean(1)
            loss = er.mean()
            if ep >= wu3:
                melu.eval()
                with torch.no_grad(): er_all = melu.recon_err(Xi_tr_t, ae).numpy()
                py = torch.tensor((er_all > np.percentile(er_all, PCT_PSEUDO)).astype(np.float32))
                melu.train()
                em, eM = er.detach().min(), er.detach().max()
                pb = ((er - em) / (eM - em + 1e-8)).clamp(1e-6, 1-1e-6)
                loss = loss + LAM_BCE * F.binary_cross_entropy(pb, py[perm[i:i+64]])
            opt3.zero_grad(); loss.backward(); opt3.step()

    # Score
    melu.eval(); melu.gate_on = True
    scores = melu.recon_err(X_te_t, ae).numpy()
    auroc  = float(roc_auc_score(y_test, scores))
    thresh = np.sort(scores)[::-1][n_anom - 1]
    y_pred = (scores >= thresh).astype(int)
    f1    = float(f1_score(y_test, y_pred, zero_division=0))
    prec  = float(precision_score(y_test, y_pred, zero_division=0))
    rec   = float(recall_score(y_test, y_pred, zero_division=0))
    return {"auroc": auroc, "f1": f1, "precision": prec, "recall": rec,
            "gate_pct": gate_pct, "n_tr": n_tr, "n_anom": n_anom}

print("run_melu_zong() defined")
print("F1 threshold: flag exactly K samples, K = len(X_outlier) (Zong 2018 rule)")


run_melu_zong() defined
F1 threshold: flag exactly K samples, K = len(X_outlier) (Zong 2018 rule)


## Cell 5 — Run experiments

> **Runtime:** ~5 min (Thyroid) to ~20 min (KDD) per dataset
> Arrhythmia ~10 min · KDDRev ~20 min

In [ ]:
DATASETS = []
if "Xi_arr" in dir():  DATASETS.append(("Arrhythmia", Xi_arr, Xo_arr, N_RUNS_SMALL, arr_src))
if "Xi_thy" in dir():  DATASETS.append(("Thyroid",    Xi_thy, Xo_thy, N_RUNS_SMALL, thy_src))
if "kdd_ok" in dir() and kdd_ok:     DATASETS.append(("KDD",    Xi_kdd, Xo_kdd, N_RUNS_LARGE, "sklearn"))
if "kddrev_ok" in dir() and kddrev_ok: DATASETS.append(("KDDRev", Xi_rev, Xo_rev, N_RUNS_LARGE, "sklearn"))

our_results = {}; our_meta = {}

for ds_name, Xi_raw, Xo_raw, n_runs, src in DATASETS:
    dim = Xi_raw.shape[1]; our_meta[ds_name] = (dim, src, len(Xi_raw), len(Xo_raw))
    print(f"\n{'='*55}\n{ds_name}  dim={dim}  n_in={len(Xi_raw)}  n_out={len(Xo_raw)}  runs={n_runs}  src={src}\n{'='*55}")
    sc = StandardScaler().fit(Xi_raw)
    Xi_sc = sc.transform(Xi_raw); Xo_sc = sc.transform(Xo_raw)
    runs = []; t0 = time.time()
    for seed in range(n_runs):
        try:
            r = run_melu_zong(Xi_sc, Xo_sc, seed=seed); runs.append(r)
        except Exception as e:
            runs.append({"auroc":0.5,"f1":0.0,"precision":0.0,"recall":0.0,"gate_pct":0.0,"n_tr":0,"n_anom":0})
        if (seed+1) % 5 == 0 or seed == n_runs-1:
            mu_f1=np.mean([r["f1"] for r in runs])*100
            mu_au=np.mean([r["auroc"] for r in runs])
            mu_gp=np.mean([r["gate_pct"] for r in runs])
            print(f"  Run {seed+1:>2}/{n_runs}  F1={mu_f1:.1f}%  AUROC={mu_au:.4f}  gate={mu_gp:.0%}")
    our_results[ds_name] = runs
    print(f"  FINAL  F1={np.mean([r['f1'] for r in runs])*100:.1f}+/-{np.std([r['f1'] for r in runs])*100:.1f}%  "
          f"AUROC={np.mean([r['auroc'] for r in runs]):.4f}  [{time.time()-t0:.0f}s]")
print("\nAll datasets complete.")



Arrhythmia  dim=274  n_in=386  n_out=66  runs=20  src=real
  Run  5/20  F1=99.4%  AUROC=1.0000  gate=52%
  Run 10/20  F1=99.5%  AUROC=1.0000  gate=52%
  Run 15/20  F1=99.7%  AUROC=1.0000  gate=52%
  Run 20/20  F1=99.7%  AUROC=1.0000  gate=52%
  FINAL  F1=99.7+/-0.6%  AUROC=1.0000  [480s]

Thyroid  dim=6  n_in=7018  n_out=182  runs=20  src=real
  Run  5/20  F1=76.6%  AUROC=0.9454  gate=48%
  Run 10/20  F1=75.1%  AUROC=0.9494  gate=48%
  Run 15/20  F1=75.3%  AUROC=0.9500  gate=48%
  Run 20/20  F1=75.7%  AUROC=0.9502  gate=47%
  FINAL  F1=75.7+/-3.7%  AUROC=0.9502  [8292s]

KDD  dim=99  n_in=25000  n_out=3377  runs=10  src=sklearn


## Cell 6 — Comparison table with published baselines

In [ ]:
PUBLISHED = {
    "Arrhythmia": {"DAGMM":49.8, "GOAD":62.8, "NeuTraL-AD":51.0, "ICL":67.2},
    "Thyroid":    {"DAGMM":93.7, "GOAD":85.3, "NeuTraL-AD":76.1, "ICL":93.8},
    "KDD":        {"DAGMM":93.7, "GOAD":99.0, "NeuTraL-AD":99.4, "ICL":99.9},
    "KDDRev":     {"DAGMM":86.6, "GOAD":97.8, "NeuTraL-AD":98.5, "ICL":98.9},
}
REFS = {
    "DAGMM":      "Zong et al. ICLR 2018, Table 2",
    "GOAD":       "Bergman & Hoshen ICLR 2020, Table 1/3",
    "NeuTraL-AD": "Qiu et al. ICML 2021, Table 3",
    "ICL":        "Shenkar & Wolf ICLR 2022, Table 2",
}
BASELINES = ["DAGMM","GOAD","NeuTraL-AD","ICL"]

our_f1   = {ds: np.mean([r["f1"]    for r in runs])*100 for ds,runs in our_results.items()}
our_f1s  = {ds: np.std( [r["f1"]    for r in runs])*100 for ds,runs in our_results.items()}
our_au   = {ds: np.mean([r["auroc"] for r in runs])     for ds,runs in our_results.items()}
our_aus  = {ds: np.std( [r["auroc"] for r in runs])     for ds,runs in our_results.items()}
our_gate = {ds: np.mean([r["gate_pct"] for r in runs])  for ds,runs in our_results.items()}

print("="*75)
print("COMPARISON TABLE -- F1 Score (%)")
print("Zong 2018 protocol | threshold = true anomaly count | 50/50 inlier split")
print("="*75)
hdr = f"  {'Dataset':<14}" + "".join(f"  {b[:12]:>12}" for b in BASELINES) + f"  {'MELU-Dt (ours)':>16}"
print(hdr); print("-"*75)
for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds not in our_results: continue
    pub = PUBLISHED[ds]; f1 = our_f1[ds]; fs = our_f1s[ds]
    best_all = max(list(pub.values()) + [f1])
    flag = "BEST" if f1 >= best_all - 0.5 else "    "
    src  = our_meta[ds][1]; warn = " SIM" if src=="simulated" else "    "
    row  = f"  {ds:<14}" + "".join(f"  {'*' if v==max(pub.values()) else ' '}{v:>10.1f}%" for b,v in pub.items())
    row += f"  {flag} {f1:>6.1f}+/-{fs:.1f}%{warn}"
    print(row)
print("-"*75)
print()
print("* = best published for that dataset")
print("BEST = MELU-Dt matches or leads best published (within 0.5%)")
print("SIM  = simulated data -- NOT comparable to published numbers")
print("       Install PyOD (pip install pyod) or place .mat files in data/")
print()
print("AUROC (ours only):")
for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds not in our_results: continue
    src = our_meta[ds][1]; warn = " SIM" if src=="simulated" else ""
    print(f"  {ds:<14}  AUROC={our_au[ds]:.4f}+/-{our_aus[ds]:.4f}  "
          f"F1={our_f1[ds]:.1f}+/-{our_f1s[ds]:.1f}%  gate={our_gate[ds]:.0%}{warn}")
print()
for b,ref in REFS.items(): print(f"  [{b}] {ref}")


## Cell 7 — Figure

In [ ]:
available = [ds for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"] if ds in our_results]
if not available:
    print("Run Cell 5 first.")
else:
    COLORS = {"DAGMM":"#534AB7","GOAD":"#BA7517","NeuTraL-AD":"#D85A30","ICL":"#888780","MELU-Dt":"#1D9E75"}
    all_m = BASELINES + ["MELU-Dt"]
    nd = len(available)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("MELU-Dt vs Published Baselines -- F1 Score\nZong 2018 Protocol: 50% normal train | 50% normal + all anomalies test", fontsize=11)

    ax = axes[0]; x = np.arange(nd); w = 0.13; offs = np.linspace(-2, 2, 5)
    for i, method in enumerate(all_m):
        vals=[]; yerrs=[]
        for ds in available:
            if method == "MELU-Dt":
                vals.append(our_f1[ds]); yerrs.append(our_f1s[ds])
            else:
                vals.append(PUBLISHED[ds][method]); yerrs.append(0)
        lw = 2.0 if method=="MELU-Dt" else 0.5
        ec = "#085041" if method=="MELU-Dt" else "none"
        ax.bar(x+offs[i]*w, vals, width=w, color=COLORS[method], alpha=0.88,
               label="MELU-Δt" if method=="MELU-Dt" else method, linewidth=lw, edgecolor=ec)
        ax.errorbar(x+offs[i]*w, vals, yerr=yerrs, fmt="none", ecolor="black", capsize=2, lw=0.8)
    for xi, ds in enumerate(available):
        ax.text(xi+offs[4]*w, our_f1[ds]+0.8, f"{our_f1[ds]:.0f}", ha="center", fontsize=8, color="#085041", fontweight="bold")
        if our_meta[ds][1] == "simulated":
            ax.text(xi, 2, "SIM", ha="center", fontsize=7, color="#BA7517")
    ax.set_xticks(x); ax.set_xticklabels(available, fontsize=11)
    ax.set_ylabel("F1 Score (%)", fontsize=11); ax.set_ylim(0, 115)
    ax.set_title(f"F1 comparison ({N_RUNS_SMALL}/{N_RUNS_LARGE} runs)", fontsize=10)
    ax.legend(fontsize=8, ncol=3); ax.grid(axis="y", alpha=0.25)

    ax = axes[1]
    deltas = [our_f1[ds] - max(PUBLISHED[ds].values()) for ds in available]
    bars = ax.bar(available, deltas, color=["#1D9E75" if d>=0 else "#D85A30" for d in deltas], alpha=0.85, width=0.5)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(1.0, color="#1D9E75", lw=1, ls="--", alpha=0.5)
    ax.set_ylabel("MELU-Δt F1 - Best Published F1 (%)", fontsize=11)
    ax.set_title("Advantage over best published baseline\n(positive = MELU-Δt leads)", fontsize=10)
    ax.grid(axis="y", alpha=0.25)
    for b, d in zip(bars, deltas):
        clr = "#085041" if d>=0 else "#993C1D"
        ax.text(b.get_x()+b.get_width()/2, d+(0.4 if d>=0 else -1.2),
                f"{d:>+.1f}%", ha="center", fontsize=11, fontweight="bold", color=clr)

    plt.tight_layout()
    plt.savefig("outputs/nb17_published_comparison.png", dpi=150, bbox_inches="tight")
    plt.show(); print("Saved: outputs/nb17_published_comparison.png")


## Cell 8 — Export CSV

In [ ]:
rows = []
for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds in PUBLISHED:
        for bl in BASELINES:
            rows.append({"dataset":ds,"method":bl,"type":"published",
                         "f1_mean":PUBLISHED[ds][bl],"f1_std":None,
                         "auroc_mean":None,"auroc_std":None,
                         "n_runs":None,"source":REFS[bl]})
    if ds in our_results:
        src = our_meta[ds][1]
        note = "" if src=="real" else " [SIMULATED -- not for publication]"
        rows.append({"dataset":ds,"method":"MELU-Dt","type":f"our_runs_{src}",
                     "f1_mean":round(our_f1[ds],2),"f1_std":round(our_f1s[ds],2),
                     "auroc_mean":round(our_au[ds],4),"auroc_std":round(our_aus[ds],4),
                     "n_runs":len(our_results[ds]),"source":"this paper"+note})
pd.DataFrame(rows).to_csv("outputs/nb17_results.csv", index=False)
print("Saved: outputs/nb17_results.csv")
print()
print("Final table:")
print(f"  {'Dataset':<14} {'DAGMM':>8} {'GOAD':>8} {'NeuTraL':>9} {'ICL':>8}  MELU-Dt (ours)")
print("  " + "-"*62)
for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds not in our_results: continue
    pub=PUBLISHED[ds]; f1=our_f1[ds]; fs=our_f1s[ds]
    best_all = max(list(pub.values())+[f1])
    flag = "BEST" if f1>=best_all-0.5 else "    "
    src=our_meta[ds][1]; warn=" SIM" if src=="simulated" else ""
    line = f"  {ds:<14}" + "".join(f" {'*' if v==max(pub.values()) else ' '}{v:>6.1f}% " for b,v in pub.items())
    line += f"  {flag} {f1:>6.1f}+/-{fs:.1f}%{warn}"
    print(line)
